# Early Stage Diabetes Risk Prediction

**Murlidhar Ravi Geetha Varma — 2025AC05598**
Machine Learning, Assignment 2

Implementing 5 classification models compared on the UCI Early Stage Diabetes Risk Prediction dataset
(ID 529): logistic regression, a decision tree, k-nearest neighbours, Gaussian naive Bayes,
and a random forest. Each is scored on accuracy, AUC, precision, recall, F1 and Matthews
correlation coefficient.

Providing the UCI link below for more details on the dataset:
https://archive.ics.uci.edu/dataset/529/early+stage+diabetes+risk+prediction+dataset

In [2]:
import sys
from pathlib import Path

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11, "axes.labelsize": 9})
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)
ACCENT = "#D85A30"

# To avoid hardcoding of src path and survives the notebook being moved.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

## 1. Exploratory data analysis

### 1.1 Structure and completeness

In [3]:
from src.config import FEATURES, SYMPTOM_FEATURES, TARGET
from src.data import load_data

raw = load_data()

print(f"{raw.shape[0]} instances, {raw.shape[1] - 1} features, 1 target")
print(f"Missing values: {int(raw.isna().sum().sum())}")
print(f"Age range: {raw['Age'].min()}-{raw['Age'].max()}")
raw.head()

520 instances, 16 features, 1 target
Missing values: 0
Age range: 16-90


,Age,Gender,Polyuria,Polydipsia,sudden weight loss,weakness,Polyphagia,Genital thrush,visual blurring,Itching,Irritability,delayed healing,partial paresis,muscle stiffness,Alopecia,Obesity,class
0,40,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes,Positive
1,58,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No,Positive
2,41,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No,Positive
3,45,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No,Positive
4,60,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Positive


520 instances and 16 features (statisfies assignment minimums of 500 and 12), but with little margin — no feature is dropped anywhere below. 
- 1 feature is continuous (`Age`), 
- 1 is Male/Female (`Gender`), 
- and the remaining 14 are Yes/No symptom indicators. 
- Nothing is missing.

### 1.2 Class distribution

In [4]:
counts = raw[TARGET].value_counts()
baseline = counts.max() / len(raw)

print(counts.to_string())
print(f"\nMajority class: {baseline:.4f} of records")
print(f"Always predicting '{counts.idxmax()}' therefore scores {baseline:.4f} accuracy")

class
Positive    320
Negative    200

Majority class: 0.6154 of records
Always predicting 'Positive' therefore scores 0.6154 accuracy


The classes are uneven. Any accuracy figure below is read against that 61.5% (if model predicts the majority class unconditionally attains that accuracy ;))

### 1.3 Duplicate records

The finding that determines how every model below is evaluated.

In [4]:
duplicated = int(raw.duplicated().sum())
distinct = raw.drop_duplicates()

# If a repeated response profile carried both labels, the repetition would be
# contradictory labelling, which needs a different remedy from leakage.
conflicting = int((raw.groupby(FEATURES, observed=True)[TARGET].nunique() > 1).sum())

print(f"Exact duplicate rows:      {duplicated} of {len(raw)}")
print(f"Distinct response profiles: {len(distinct)}")
print(f"Profiles with both labels:  {conflicting}")
print(f"\nClass balance after deduplication: "
      f"{distinct[TARGET].value_counts(normalize=True).max():.4f} majority")

Exact duplicate rows:      269 of 520
Distinct response profiles: 251
Profiles with both labels:  0

Class balance after deduplication: 0.6892 majority


More than half the records repeat another record exactly, leaving 251 distinct response
profiles. No profile carries both labels, so this is repetition rather than inconsistent
labelling.

The consequence is that a conventional `train_test_split` places identical rows on both sides
of the split. A model that memorises a training row then scores it correctly in the test set
without having generalised at all, and the high accuracies commonly reported on this dataset
partly reflect that. Deduplication before splitting is therefore have to be applied throughout

Deduplication also worsens the imbalance, from 61.5% to 68.9% majority, which raises the
accuracy floor the models must clear.